In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.sql import text
import warnings
import sys
import os # Esto sube un nivel en las carpetas para encontrar la raíz del proyecto
from dotenv import load_dotenv

# Verificar dónde está buscando el .env
print("Directorio actual:", os.getcwd())

# Cargar y comprobar variables
load_dotenv("../.env")

Directorio actual: h:\Cursos\Adalab_Analista & IA\taller_git\e-commerce-operations-insights\notebooks


True

In [2]:
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

In [3]:
df_data = pd.read_csv("../files/processed/dataco_supply_chain.csv", sep=None, engine="python", encoding="latin-1")

In [4]:
df_data.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime
157228,TRANSFER,0,0,39.29,120.89,Shipping on time,0,18,Men's Footwear,Caguas,Puerto Rico,Gerald,6318,Garza,Consumer,PR,5013 Tawny Cider Meadow,725,4,Apparel,18.238152,-66.370567,Orizaba,MÃ©xico,6318,2017-02-08 04:00:00,52692,403,9.10,0.07,131725,129.99,0.33,1,129.99,120.89,39.29,Central America,PROCESSING,403,18,Nike Men's CJ Elite 2 TD Football Cleat,129.99,Same Day,02-08-2017,16:00
22704,DEBIT,4,4,-27.78,149.38,Shipping on time,0,17,Cleats,Caguas,Puerto Rico,Harold,10353,Taylor,Consumer,PR,851 Green Cider Concession,725,4,Apparel,18.226589,-66.370537,Sale,Marruecos,10353,2016-11-25 01:49:00,47548,365,30.59,0.17,118867,59.99,-0.19,3,179.97,149.38,-27.78,North Africa,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,Standard Class,11-29-2016,01:49
158522,TRANSFER,6,2,82.80,180.00,Late delivery,1,24,Women's Apparel,Caguas,Puerto Rico,Walter,1624,Huerta,Consumer,PR,8081 Cozy Edge,725,5,Golf,18.266869,-66.370583,Rabat,Marruecos,1624,2016-09-24 08:34:00,43320,502,20.00,0.10,108225,50.00,0.46,4,200.00,180.00,82.80,North Africa,PROCESSING,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Second Class,09-30-2016,08:34
97567,DEBIT,3,2,27.36,56.99,Late delivery,1,17,Cleats,Columbus,EE. UU.,Bruce,3081,Thornton,Home Office,OH,1763 Dusty Trace,43230,4,Apparel,40.023075,-82.873261,Santo Domingo,RepÃºblica Dominicana,3081,2015-05-03 13:29:00,8397,365,3.00,0.05,20972,59.99,0.48,1,59.99,56.99,27.36,Caribbean,COMPLETE,365,17,Perfect Fitness Perfect Rip Deck,59.99,Second Class,05-06-2015,13:29
91332,PAYMENT,6,4,18.49,147.94,Late delivery,1,67,DVDs,Caguas,Puerto Rico,Miranda,18603,Mathis,Corporate,PR,1975 Crystal Leaf Autoroute,725,9,Discs Shop,18.289145,-66.037056,Kunming,China,18603,2017-12-31 13:00:00,75050,1354,16.44,0.10,178365,164.38,0.13,1,164.38,147.94,18.49,Eastern Asia,PENDING_PAYMENT,1354,67,DVDs,164.38,Standard Class,01-06-2018,13:00


In [5]:
dim_products = df_data[['product_card_id', 'product_name', 'product_price', 
                        'category_id', 'category_name', 'department_id', 
                        'department_name']].drop_duplicates(subset=['product_card_id']).reset_index(drop=True)

In [6]:
dim_products.tail(5)

,product_card_id,product_name,product_price,category_id,category_name,department_id,department_name
113,646,Merrell Women's Grassbow Sport Hiking Shoe,99.99,30,Men's Golf Clubs,6,Outdoors
114,1361,Toys,11.54,74,Toys,7,Fan Shop
115,1073,Pelican Sunstream 100 Kayak,199.99,48,Water Sports,7,Fan Shop
116,1059,Pelican Maverick 100X Kayak,349.99,48,Water Sports,7,Fan Shop
117,1014,O'Brien Men's Neoprene Life Vest,49.98,46,Indoor/Outdoor Games,7,Fan Shop


In [7]:
dim_customers = df_data[['customer_id', 'customer_fname', 'customer_lname', 
                    'customer_segment', 'customer_street', 
                    'customer_city', 'customer_state', 'customer_zipcode', 'customer_country'
                    ]].drop_duplicates(subset=['customer_id']).reset_index(drop=True)

In [8]:
dim_customers.head(5)

,customer_id,customer_fname,customer_lname,customer_segment,customer_street,customer_city,customer_state,customer_zipcode,customer_country
0,20755,Cally,Holloway,Consumer,5365 Noble Nectar Island,Caguas,PR,725,Puerto Rico
1,19492,Irene,Luna,Consumer,2679 Rustic Loop,Caguas,PR,725,Puerto Rico
2,19491,Gillian,Maldonado,Consumer,8510 Round Bear Gate,San Jose,CA,95125,EE. UU.
3,19490,Tana,Tate,Home Office,3200 Amber Bend,Los Angeles,CA,90027,EE. UU.
4,19489,Orli,Hendricks,Corporate,8671 Iron Anchor Corners,Caguas,PR,725,Puerto Rico


In [9]:
dim_stores = df_data[['latitude', 'longitude', 'shipping_mode']].drop_duplicates().reset_index(drop=True)
dim_stores['store_id'] = dim_stores.index + 1
dim_stores = dim_stores[['store_id', 'latitude', 'longitude', 'shipping_mode']]

In [10]:
dim_stores.sample(5)

,store_id,latitude,longitude,shipping_mode
216,217,25.825075,-80.195885,Standard Class
8922,8923,41.053909,-73.548370,First Class
2258,2259,18.294447,-66.370544,Standard Class
14385,14386,18.359024,-66.078148,Standard Class
16655,16656,18.262520,-66.370514,Standard Class


In [11]:
fact_sales = pd.merge(df_data, dim_stores, on=['latitude', 'longitude', 'shipping_mode'], how='left')

In [12]:
fact_sales.sample(5)

,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,customer_country,customer_fname,customer_id,customer_lname,customer_segment,customer_state,customer_street,customer_zipcode,department_id,department_name,latitude,longitude,order_city,order_country,order_customer_id,order_date_dateorders,order_id,order_item_cardprod_id,order_item_discount,order_item_discount_rate,order_item_id,order_item_product_price,order_item_profit_ratio,order_item_quantity,sales,order_item_total,order_profit_per_order,order_region,order_status,product_card_id,product_category_id,product_name,product_price,shipping_mode,date,datetime,store_id
165041,DEBIT,3,4,53.29,163.96,Advance shipping,0,29,Shop By Sport,Newark,EE. UU.,Amber,5338,Smith,Home Office,NJ,2221 Lazy Sky Trail,7104,5,Golf,40.132694,-74.060295,CuliacÃ¡n,MÃ©xico,5338,2015-04-23 20:30:00,7732,627,35.99,0.18,19333,39.99,0.33,5,199.95,163.96,53.29,Central America,COMPLETE,627,29,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class,04-26-2015,20:30,16202
77444,PAYMENT,5,4,39.34,122.95,Late delivery,1,46,Indoor/Outdoor Games,Tempe,EE. UU.,Brenda,3773,Prince,Consumer,AZ,9453 High Concession,85283,7,Fan Shop,33.357388,-111.955185,Fontainebleau,Francia,3773,2017-09-11 08:47:00,67434,1014,26.99,0.18,168554,49.98,0.32,3,149.94,122.95,39.34,Western Europe,PENDING_PAYMENT,1014,46,O'Brien Men's Neoprene Life Vest,49.98,Standard Class,09-16-2017,08:47,4653
35063,CASH,2,4,-192.13,116.44,Advance shipping,0,37,Electronics,Cleveland,EE. UU.,Christina,7423,Smith,Consumer,OH,3481 Wishing Port,44102,6,Outdoors,41.480122,-81.735077,Tilburg,PaÃ­ses Bajos,7423,2015-10-08 03:06:00,19191,835,11.52,0.09,47941,31.99,-1.65,4,127.96,116.44,-192.13,Western Europe,CLOSED,835,37,Bridgestone e6 Straight Distance NFL Carolina,31.99,Standard Class,10-10-2015,03:06,1969
178417,PAYMENT,2,1,30.34,142.44,Late delivery,1,46,Indoor/Outdoor Games,Caguas,Puerto Rico,Kelly,5715,Smith,Corporate,PR,9623 Stony Bay,725,7,Fan Shop,18.237514,-66.370537,Montevideo,Uruguay,5715,2015-05-05 15:56:00,8541,1014,7.50,0.05,21313,49.98,0.21,3,149.94,142.44,30.34,South America,PENDING_PAYMENT,1014,46,O'Brien Men's Neoprene Life Vest,49.98,First Class,05-07-2015,15:56,16761
81454,DEBIT,6,2,118.75,237.50,Late delivery,1,24,Women's Apparel,Chesapeake,EE. UU.,Carolyn,1290,Charles,Consumer,VA,7810 Quaking Terrace,23322,5,Golf,36.705250,-76.243279,Rillieux-la-Pape,Francia,1290,2017-09-06 03:00:00,67075,502,12.50,0.05,167666,50.00,0.50,5,250.00,237.50,118.75,Western Europe,ON_HOLD,502,24,Nike Men's Dri-FIT Victory Golf Polo,50.00,Second Class,09-12-2017,03:00,15894


In [13]:
fact_sales = fact_sales[[
    'order_item_id', 'order_id', 'customer_id', 'product_card_id', 'store_id',
    'order_date_dateorders',
    'type', 'order_status', 'days_for_shipping_real', 
    'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk',
    'order_city', 'order_country', 'order_region',
    'sales', 'order_item_quantity', 'order_item_product_price', 
    'order_item_discount', 'order_item_discount_rate', 'order_item_total', 
    'order_item_profit_ratio', 'benefit_per_order', 'order_profit_per_order', 'sales_per_customer'
]]
 
print("¡DataFrames del Modelo en Estrella listos en memoria!")

¡DataFrames del Modelo en Estrella listos en memoria!


In [14]:
import os

from dotenv import load_dotenv

from sqlalchemy import create_engine, text
 
load_dotenv()
 
USER    = os.getenv("DB_USER")
PASS    = os.getenv("DB_PASSWORD")
HOST    = os.getenv("DB_HOST")
PORT    = os.getenv("DB_PORT", "3306")   # 3306 como valor por defecto

DB_NAME = os.getenv("DB_NAME")
 
# 1️⃣ Conectar al servidor SIN base de datos

engine_server = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}") 
# 2️⃣ Crear la base de datos si no existe

with engine_server.begin() as con:
    con.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}"))
    print(f"✅ Base de datos '{DB_NAME}' lista")
 
engine_server.dispose()
 
# 3️⃣ Conectar ahora CON la base de datos

engine = create_engine(f"mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{DB_NAME}")
print(f"✅ Conectado a '{DB_NAME}'")
 

✅ Base de datos 'dataco' lista
✅ Conectado a 'dataco'


In [16]:
# Carga del DataFrame en MySQL
# Esto crea automáticamente la tabla products en la base de datos
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
'''# Comprobación previa para evitar errores al asignar la clave primaria
dim_products = dim_products.dropna(subset=["Employee_Number"])
dim_products = dim_products.drop_duplicates(subset=["Employee_Number"])'''
 
dim_products.to_sql("products", con=engine, if_exists="replace", index=False)
 
# Asignación de la clave primaria una vez creada la tabla
with engine.begin() as con:
    con.execute(text("""
        ALTER TABLE products
        MODIFY product_card_id INT NOT NULL"""))
    con.execute(text("""
        ALTER TABLE products
        ADD PRIMARY KEY (product_card_id)"""))
 
print("¡Listo! Base de datos, tabla y clave primaria creadas correctamente.")

¡Listo! Base de datos, tabla y clave primaria creadas correctamente.
